# Fase 3 - Analitica Supervisada (Spark MLlib)

Regresion para predecir `Transfer Value` y clasificacion para el potencial de jovenes talentos (proyeccion via `Caps`).

In [1]:
import os
os.environ.setdefault('JAVA_HOME', '/opt/homebrew/opt/openjdk@17')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = SparkSession.builder \
    .appName('ScoutingSupervised') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

df = spark.read.parquet('../data/processed/players_clean.parquet')

attribute_cols = [
    'Acc','Wor','Vis','Thr','Tec','Tea','Tck','Str','Sta','TRO','Ref','Pun','Pos','Pen','Pas','Pac',
    '1v1','OtB','Forma_Fisica_Natural','Mar','L Th','Lon','Ldr','Kic','Jum','Hea','Han','Fre','Fla','Fir','Fin','Ecc',
    'Dri','Det','Dec','Cro','Cor','Cnt','Cmp','Com','Cmd','Bra','Bal','Ant','Agi','Agg','Aer','Vers',
    'Temp','Spor','Prof','Pres','Loy','Inj Pr','Imp M','Dirt','Amb','Ada','Cons','Cont',
]
feature_cols = attribute_cols + ['Age', 'Height_cm', 'Weight_kg']
print(len(feature_cols), 'features')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/15 15:57:44 WARN Utils: Your hostname, MacBook-de-Rodrigo.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.123 instead (on interface en0)
26/07/15 15:57:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/15 15:57:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


63 features


## Parte A - Regresion: prediccion de Transfer Value

Se excluyen los jugadores `Not for Sale` (sin valor de mercado definido, ver Fase 1) del entrenamiento y evaluacion de la regresion.

In [2]:
df_reg = df.filter(F.col('Transfer_Value_num').isNotNull()).na.drop(subset=feature_cols)
print(df_reg.count())

train_reg, test_reg = df_reg.randomSplit([0.8, 0.2], seed=42)
print(train_reg.count(), test_reg.count())

26/07/15 15:57:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


89620


71792 17828


In [3]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)

lr = LinearRegression(featuresCol='features', labelCol='Transfer_Value_num', maxIter=100)
gbt = GBTRegressor(featuresCol='features', labelCol='Transfer_Value_num', maxIter=50, maxDepth=5, seed=42)

pipeline_lr = Pipeline(stages=[assembler, scaler, lr])
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

reg_evaluator_rmse = RegressionEvaluator(labelCol='Transfer_Value_num', predictionCol='prediction', metricName='rmse')
reg_evaluator_mae = RegressionEvaluator(labelCol='Transfer_Value_num', predictionCol='prediction', metricName='mae')
reg_evaluator_r2 = RegressionEvaluator(labelCol='Transfer_Value_num', predictionCol='prediction', metricName='r2')

results_reg = {}
for name, pipeline in [('LinearRegression', pipeline_lr), ('GBTRegressor', pipeline_gbt)]:
    model = pipeline.fit(train_reg)
    preds_train = model.transform(train_reg)
    preds_test = model.transform(test_reg)
    results_reg[name] = {
        'rmse_train': reg_evaluator_rmse.evaluate(preds_train),
        'rmse_test': reg_evaluator_rmse.evaluate(preds_test),
        'mae_test': reg_evaluator_mae.evaluate(preds_test),
        'r2_test': reg_evaluator_r2.evaluate(preds_test),
        'model': model,
    }
    print(name, {k: v for k, v in results_reg[name].items() if k != 'model'})

26/07/15 15:57:52 WARN Instrumentation: [c4b1f403] regParam is zero, which might cause numerical instability and overfitting.


26/07/15 15:57:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


26/07/15 15:57:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


LinearRegression {'rmse_train': 6508022.732057086, 'rmse_test': 6057292.069115672, 'mae_test': 1916914.3250995034, 'r2_test': 0.12232531708284688}


GBTRegressor {'rmse_train': 2141679.1445515486, 'rmse_test': 5652931.270384948, 'mae_test': 880686.7250823036, 'r2_test': 0.2355942385169021}


`rmse_train` vs `rmse_test` cercanos indica que no hay overfitting severo. Si `rmse_train << rmse_test`, el modelo esta memorizando y se debe reducir `maxDepth` / aumentar regularizacion.

## Ajuste de hiperparametros con CrossValidator (GBTRegressor)

In [4]:
param_grid_gbt = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5, 7])
    .addGrid(gbt.maxIter, [30, 50])
    .build())

cv_gbt = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid_gbt,
    evaluator=reg_evaluator_rmse,
    numFolds=3,
    parallelism=2,
    seed=42,
)

cv_gbt_model = cv_gbt.fit(train_reg)
best_gbt_preds = cv_gbt_model.transform(test_reg)
print('RMSE test (mejor GBT via CV):', reg_evaluator_rmse.evaluate(best_gbt_preds))
print('R2 test (mejor GBT via CV):', reg_evaluator_r2.evaluate(best_gbt_preds))

26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1000.1 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1000.9 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1002.9 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1006.0 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1010.6 KiB


26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1011.3 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1011.8 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1012.8 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1013.6 KiB


26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1015.9 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1020.4 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1027.7 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1028.6 KiB


26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1029.1 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1030.1 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1030.9 KiB
26/07/15 15:59:10 WARN DAGScheduler: Broadcasting large task binary with size 1033.3 KiB


26/07/15 15:59:11 WARN DAGScheduler: Broadcasting large task binary with size 1037.8 KiB
26/07/15 15:59:11 WARN DAGScheduler: Broadcasting large task binary with size 1045.7 KiB
26/07/15 15:59:11 WARN DAGScheduler: Broadcasting large task binary with size 1048.0 KiB


...

26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1098.5 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1100.2 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1103.7 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1106.2 KiB


26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1106.6 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1107.7 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1108.4 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1110.5 KiB
26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1114.5 KiB


26/07/15 16:00:57 WARN DAGScheduler: Broadcasting large task binary with size 1122.0 KiB


RMSE test (mejor GBT via CV): 5200190.35240981


R2 test (mejor GBT via CV): 0.35313295886579743


## Importancia de variables (GBTRegressor)

In [5]:
best_gbt_model = cv_gbt_model.bestModel.stages[-1]
importances = list(zip(feature_cols, best_gbt_model.featureImportances.toArray()))
importances.sort(key=lambda x: -x[1])
for feat, imp in importances[:15]:
    print(f"{feat}: {imp:.4f}")

Pac: 0.1389
Cmp: 0.0783
Sta: 0.0644
Acc: 0.0555
Tec: 0.0531
Ant: 0.0475
Fir: 0.0453
Dec: 0.0391
Cro: 0.0362
Vis: 0.0345
Pres: 0.0326
Tea: 0.0311
Str: 0.0248
Wor: 0.0246
Dri: 0.0240


### Traduccion a impacto de negocio (regresion)

`Pac`, `Cmp`, `Tec`, `Dec`, `Ant` (velocidad, compostura, tecnica, decisiones, anticipacion) son los
principales drivers del valor segun el modelo — coincide con la lectura de correlaciones de la Fase 1
(atributos mentales/tecnicos por encima de fisicos puros). Para dimensionar el error en terminos que el comite
entienda, se expresa el RMSE/MAE como porcentaje del valor promedio de transferencia.

In [6]:
mean_value_reg = df_reg.select(F.mean('Transfer_Value_num')).first()[0]
median_value_reg = df_reg.approxQuantile('Transfer_Value_num', [0.5], 0.01)[0]
rmse_best = reg_evaluator_rmse.evaluate(best_gbt_preds)
mae_best = reg_evaluator_mae.evaluate(best_gbt_preds)
r2_best = reg_evaluator_r2.evaluate(best_gbt_preds)

print(f"Valor promedio de transferencia: ${mean_value_reg:,.0f}  |  Valor mediano: ${median_value_reg:,.0f}")
print(f"RMSE test (GBT + CV): ${rmse_best:,.0f}")
print(f"MAE test (GBT + CV): ${mae_best:,.0f}  ({mae_best / mean_value_reg * 100:.0f}% del valor promedio)")
print(f"R2 test (GBT + CV): {r2_best:.3f}")

# El mercado es muy asimetrico (mediana << promedio): el error absoluto tipico depende mucho
# del rango de precio del jugador, asi que se reporta el MAE segmentado por tramo de valor.
error_por_tramo = best_gbt_preds.withColumn(
    'tramo_valor',
    F.when(F.col('Transfer_Value_num') < 1e5, '< $100K')
     .when(F.col('Transfer_Value_num') < 1e6, '$100K - $1M')
     .when(F.col('Transfer_Value_num') < 1e7, '$1M - $10M')
     .otherwise('>= $10M')
).withColumn('abs_error', F.abs(F.col('Transfer_Value_num') - F.col('prediction')))

error_por_tramo.groupBy('tramo_valor').agg(
    F.count('*').alias('n_jugadores'),
    F.round(F.mean('Transfer_Value_num'), 0).alias('valor_promedio_tramo'),
    F.round(F.mean('abs_error'), 0).alias('mae_tramo'),
).orderBy('valor_promedio_tramo').show(truncate=False)

Valor promedio de transferencia: $952,210  |  Valor mediano: $45,000
RMSE test (GBT + CV): $5,200,190
MAE test (GBT + CV): $982,482  (103% del valor promedio)
R2 test (GBT + CV): 0.353


+-----------+-----------+--------------------+-----------+
|tramo_valor|n_jugadores|valor_promedio_tramo|mae_tramo  |
+-----------+-----------+--------------------+-----------+
|< $100K    |10720      |18022.0             |358441.0   |
|$100K - $1M|5123       |375634.0            |652240.0   |
|$1M - $10M |1691       |3038831.0           |2823724.0  |
|>= $10M    |294        |3.2788776E7         |1.8900879E7|
+-----------+-----------+--------------------+-----------+



**Nota sobre RMSE/MAE en un mercado asimetrico:** la mediana de `Transfer_Value_num` (~$45K) es mucho menor
que el promedio (~$952K) porque la mayoria de los jugadores del dataset son de ligas menores o sin mercado
activo, mientras unas pocas superestrellas valen >€100M.

**Limitacion real detectada (tabla de arriba):** en terminos absolutos el error crece con el precio, como es
esperable (~$358K de error en el tramo `< $100K` vs. ~$18.9M en el tramo `>= $10M`). Pero en terminos
**relativos** el modelo es mucho peor justo en el tramo mas poblado: para el 60% de los jugadores (los de
`< $100K`, valor promedio real ~$18K), un error tipico de ~$358K representa sobrestimar su valor en ~20x. El
modelo, entrenado para minimizar el error absoluto global, esta implicitamente optimizado para acertar en el
rango medio/alto (donde esta la mayor parte de la varianza en dolares) y sacrifica precision relativa en el
extremo barato — justo el segmento donde el consorcio querria detectar "gangas" (motor de clones de la Fase
2). Para producccion, la siguiente iteracion natural seria modelar `log(Transfer_Value_num)` en vez del valor
crudo, lo que penaliza el error relativo por igual en todos los rangos de precio.

`R2 ≈ 0.35` significa que el modelo explica una parte relevante pero no toda la varianza del valor —
consistente con la Fase 1: el valor de mercado depende tambien de factores que no estan en los atributos de
habilidad (liga, marca personal, situacion contractual, edad de pico), asi que el rol del modelo es dar una
**referencia cuantitativa objetiva** para contrastar contra el criterio del director deportivo, no un precio
definitivo.

## Parte B - Clasificacion: potencial de jovenes talentos

Target: si el jugador ha sido convocado a su seleccion (`Caps > 0`), usando solo atributos de habilidad (sin `Transfer Value`, que dependeria del resultado y causaria data leakage). El modelo se entrena con todos los jugadores y se aplica luego sobre los menores de 21 anios como 'proyeccion de potencial'.

In [7]:
df_clf = df.na.drop(subset=feature_cols).withColumn(
    'label', (F.coalesce(F.col('Caps'), F.lit(0)) > 0).cast('double')
)
df_clf.groupBy('label').count().show()

train_clf, test_clf = df_clf.randomSplit([0.8, 0.2], seed=42)

+-----+-----+
|label|count|
+-----+-----+
|  1.0| 7789|
|  0.0|83882|
+-----+-----+



In [8]:
assembler_clf = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler_clf = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)

log_reg = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.01)
rf_clf = RandomForestClassifier(featuresCol='features', labelCol='label', maxDepth=6, numTrees=100, seed=42)

pipeline_logreg = Pipeline(stages=[assembler_clf, scaler_clf, log_reg])
pipeline_rf = Pipeline(stages=[assembler_clf, scaler_clf, rf_clf])

f1_evaluator = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
auc_evaluator = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')

results_clf = {}
for name, pipeline in [('LogisticRegression', pipeline_logreg), ('RandomForest', pipeline_rf)]:
    model = pipeline.fit(train_clf)
    preds_train = model.transform(train_clf)
    preds_test = model.transform(test_clf)
    results_clf[name] = {
        'f1_train': f1_evaluator.evaluate(preds_train),
        'f1_test': f1_evaluator.evaluate(preds_test),
        'auc_test': auc_evaluator.evaluate(preds_test),
        'model': model,
    }
    print(name, {k: v for k, v in results_clf[name].items() if k != 'model'})

LogisticRegression {'f1_train': 0.9031998822447747, 'f1_test': 0.9033032832900386, 'auc_test': 0.8551351920616903}


26/07/15 16:01:21 WARN DAGScheduler: Broadcasting large task binary with size 1198.1 KiB


26/07/15 16:01:23 WARN DAGScheduler: Broadcasting large task binary with size 1066.1 KiB


26/07/15 16:01:24 WARN DAGScheduler: Broadcasting large task binary with size 1066.1 KiB


26/07/15 16:01:24 WARN DAGScheduler: Broadcasting large task binary with size 1053.6 KiB


RandomForest {'f1_train': 0.889817483436376, 'f1_test': 0.8916205115556249, 'auc_test': 0.8478048735102108}


## Ajuste de hiperparametros con CrossValidator (RandomForest) y control de overfitting

In [9]:
param_grid_rf = (ParamGridBuilder()
    .addGrid(rf_clf.maxDepth, [4, 6, 8])
    .addGrid(rf_clf.numTrees, [50, 100])
    .build())

cv_rf = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=param_grid_rf,
    evaluator=f1_evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

cv_rf_model = cv_rf.fit(train_clf)
preds_train_best = cv_rf_model.transform(train_clf)
preds_test_best = cv_rf_model.transform(test_clf)
print('F1 train (mejor RF via CV):', f1_evaluator.evaluate(preds_train_best))
print('F1 test (mejor RF via CV):', f1_evaluator.evaluate(preds_test_best))
print('AUC test (mejor RF via CV):', auc_evaluator.evaluate(preds_test_best))

26/07/15 16:01:35 WARN DAGScheduler: Broadcasting large task binary with size 1227.7 KiB


26/07/15 16:01:36 WARN DAGScheduler: Broadcasting large task binary with size 1140.3 KiB


26/07/15 16:01:38 WARN DAGScheduler: Broadcasting large task binary with size 1186.5 KiB


26/07/15 16:01:39 WARN DAGScheduler: Broadcasting large task binary with size 1906.8 KiB


26/07/15 16:01:40 WARN DAGScheduler: Broadcasting large task binary with size 1227.7 KiB


...

26/07/15 16:02:27 WARN DAGScheduler: Broadcasting large task binary with size 1578.8 KiB


F1 train (mejor RF via CV): 0.902297150088188


F1 test (mejor RF via CV): 0.8971085961085489


26/07/15 16:02:28 WARN DAGScheduler: Broadcasting large task binary with size 1566.4 KiB


AUC test (mejor RF via CV): 0.8607410313737117


## Importancia de variables (RandomForest)

In [10]:
best_rf_model = cv_rf_model.bestModel.stages[-1]
importances_clf = list(zip(feature_cols, best_rf_model.featureImportances.toArray()))
importances_clf.sort(key=lambda x: -x[1])
for feat, imp in importances_clf[:15]:
    print(f"{feat}: {imp:.4f}")

Cmp: 0.0931
Bal: 0.0816
Ant: 0.0795
Amb: 0.0623
Age: 0.0559
Prof: 0.0389
Pen: 0.0376
Cnt: 0.0372
Str: 0.0316
Bra: 0.0301
Pac: 0.0273
Tec: 0.0270
Imp M: 0.0260
Tea: 0.0234
Sta: 0.0216


### Traduccion a impacto de negocio (clasificacion)

`Cmp` (compostura), `Bal` (balance), `Amb` (ambicion) y `Ant` (anticipacion) dominan la prediccion de
potencial — de nuevo, atributos mentales por encima de fisicos. Antes de aplicar el modelo a jovenes
talentos, se examina la matriz de confusion en el set de test para entender el costo de cada tipo de error:

In [11]:
cm = preds_test_best.groupBy('label', 'prediction').count().orderBy('label', 'prediction')
cm.show()

cm_pd = cm.toPandas().pivot(index='label', columns='prediction', values='count').fillna(0)
fn = cm_pd.loc[1.0, 0.0] if 0.0 in cm_pd.columns else 0
fp = cm_pd.loc[0.0, 1.0] if 1.0 in cm_pd.columns else 0
tp = cm_pd.loc[1.0, 1.0] if 1.0 in cm_pd.columns else 0
tn = cm_pd.loc[0.0, 0.0] if 0.0 in cm_pd.columns else 0
print(f"Verdaderos positivos (potencial real detectado): {tp:.0f}")
print(f"Falsos negativos (talento real que el modelo descarta): {fn:.0f}")
print(f"Falsos positivos (jugador sin potencial marcado como prospecto): {fp:.0f}")
print(f"Verdaderos negativos: {tn:.0f}")

26/07/15 16:02:29 WARN DAGScheduler: Broadcasting large task binary with size 1579.8 KiB


26/07/15 16:02:29 WARN DAGScheduler: Broadcasting large task binary with size 1473.6 KiB
26/07/15 16:02:29 WARN DAGScheduler: Broadcasting large task binary with size 1580.0 KiB


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|16692|
|  0.0|       1.0|   35|
|  1.0|       0.0| 1342|
|  1.0|       1.0|  168|
+-----+----------+-----+



Verdaderos positivos (potencial real detectado): 168
Falsos negativos (talento real que el modelo descarta): 1342
Falsos positivos (jugador sin potencial marcado como prospecto): 35
Verdaderos negativos: 16692


26/07/15 16:02:30 WARN DAGScheduler: Broadcasting large task binary with size 1474.0 KiB
26/07/15 16:02:30 WARN DAGScheduler: Broadcasting large task binary with size 1474.0 KiB
26/07/15 16:02:30 WARN DAGScheduler: Broadcasting large task binary with size 1473.9 KiB


In [12]:
precision_pos = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_pos = 2 * precision_pos * recall_pos / (precision_pos + recall_pos) if (precision_pos + recall_pos) > 0 else 0.0
print(f"Precision (clase potencial=1): {precision_pos:.3f}")
print(f"Recall (clase potencial=1): {recall_pos:.3f}")
print(f"F1 (clase potencial=1): {f1_pos:.3f}")

Precision (clase potencial=1): 0.828
Recall (clase potencial=1): 0.111
F1 (clase potencial=1): 0.196


**Hallazgo clave — el F1/AUC agregados esconden un problema real:** con precision ≈0.83 pero **recall ≈0.11**,
el modelo (con el umbral por defecto `probability >= 0.5`) solo detecta ~1 de cada 9 jugadores que realmente
llegan a tener minutos con su seleccion. El `F1=0.90` reportado antes es el promedio ponderado por clase de
`MulticlassClassificationEvaluator`, y como el 91.5% de los jugadores son `label=0`, ese numero esta dominado
por lo bien que el modelo predice "no va a tener potencial" (mayoria trivial), no por si detecta a los que si
lo tienen. Esto es una consecuencia directa del **desbalance de clases** (7,789 positivos vs. 83,882
negativos) y es exactamente el tipo de trampa que hay que exponer ante el comite en vez de reportar solo la
metrica agregada mas favorable.

**Lectura de negocio (costo de cada error):**

- **Falso positivo** (35 casos): costo acotado — un scout evalua y descarta a un jugador sin potencial real.
- **Falso negativo** (1,342 casos): costo de oportunidad alto — un talento real que el sistema no habria
  priorizado para seguimiento.

Dado el volumen de falsos negativos con el umbral por defecto, **no conviene usar 0.5 como punto de corte
operativo**. Se prueba bajar el umbral de decision para ver el trade-off real precision/recall:

In [13]:
from pyspark.ml.functions import vector_to_array

probs_test = preds_test_best.withColumn('prob_1', vector_to_array('probability')[1])

for thresh in [0.5, 0.35, 0.25, 0.15, 0.1, 0.05]:
    preds_thresh = probs_test.withColumn('pred_thresh', (F.col('prob_1') >= thresh).cast('double'))
    tp_t = preds_thresh.filter((F.col('label') == 1) & (F.col('pred_thresh') == 1)).count()
    fp_t = preds_thresh.filter((F.col('label') == 0) & (F.col('pred_thresh') == 1)).count()
    fn_t = preds_thresh.filter((F.col('label') == 1) & (F.col('pred_thresh') == 0)).count()
    precision_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0.0
    recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0
    f1_t = 2 * precision_t * recall_t / (precision_t + recall_t) if (precision_t + recall_t) > 0 else 0.0
    print(f"umbral={thresh:.2f}: precision={precision_t:.3f} recall={recall_t:.3f} f1={f1_t:.3f} "
          f"(TP={tp_t}, FP={fp_t}, FN={fn_t})")

26/07/15 16:02:30 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


26/07/15 16:02:31 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


26/07/15 16:02:31 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


umbral=0.50: precision=0.828 recall=0.111 f1=0.196 (TP=168, FP=35, FN=1342)


26/07/15 16:02:32 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


...

umbral=0.10: precision=0.248 recall=0.757 f1=0.373 (TP=1143, FP=3471, FN=367)


26/07/15 16:02:37 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


26/07/15 16:02:38 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


26/07/15 16:02:38 WARN DAGScheduler: Broadcasting large task binary with size 1578.6 KiB


umbral=0.05: precision=0.151 recall=0.934 f1=0.260 (TP=1411, FP=7930, FN=99)


**Recomendacion operativa:** bajar el umbral (ej. a ~0.15-0.20) recupera una porcion sustancial de los talentos
reales a costa de mas falsos positivos — un intercambio favorable dado que el costo de un falso positivo
(tiempo de scouting) es mucho menor que el de un falso negativo (talento perdido). Se recomienda fijar el
umbral de produccion en la zona donde el recall sube marcadamente sin que la precision colapse (ver tabla de
arriba), y de todas formas tratar la salida del modelo como una **lista corta priorizada para revision
humana**, no como una decision automatica de fichaje.

## Aplicacion: proyeccion de potencial en jovenes talentos (<= 21 anios)

In [14]:
from pyspark.ml.functions import vector_to_array

jovenes = df_clf.filter(F.col('Age') <= 21)
proyeccion = cv_rf_model.transform(jovenes).withColumn('prob_potencial', vector_to_array('probability')[1])
proyeccion.select('Name', 'Age', 'Position', 'Club', 'label', 'prediction', 'prob_potencial') \
    .orderBy(F.desc('prob_potencial')) \
    .show(15, truncate=False)

26/07/15 16:02:39 WARN DAGScheduler: Broadcasting large task binary with size 1488.7 KiB


+---------------------+---+----------------+-----------------+-----+----------+------------------+
|Name                 |Age|Position        |Club             |label|prediction|prob_potencial    |
+---------------------+---+----------------+-----------------+-----+----------+------------------+
|Erling Haaland       |21 |ST (C)          |Man City         |1.0  |1.0       |0.8856078798293546|
|Jude Bellingham      |18 |DM, M (C)       |Borussia Dortmund|1.0  |1.0       |0.8491123889781426|
|Bukayo Saka          |20 |AM (RL)         |Arsenal          |1.0  |1.0       |0.8185826663024753|
|Gavi                 |17 |DM, M/AM (C)    |Barcelona        |1.0  |1.0       |0.8143648721529273|
|Vinícius Júnior      |21 |AM (RL)         |R. Madrid        |1.0  |1.0       |0.7758769879338896|
|Charles De Ketelaere |21 |M/AM (C), ST (C)|Milan            |1.0  |1.0       |0.7755243155368813|
|Pedri                |19 |M (C), AM (RLC) |Barcelona        |1.0  |1.0       |0.7654876936609885|
|Jamal Mus

## Cierre: de metricas a decisiones de negocio

- El modelo detecta correctamente como top prospects a jugadores que hoy son estrellas confirmadas
  (Haaland, Bellingham, Saka, Gavi, Vinícius Jr., Pedri, Musiala) usando solo atributos de habilidad a una
  edad temprana — evidencia de que el enfoque cuantitativo puede anticipar lo que el ojo humano tambien
  detecto, pero de forma sistematica y escalable a miles de jugadores que ningun cuerpo de scouts humano
  podria cubrir a la vez.
- El analisis de la matriz de confusion (arriba) muestra que el umbral por defecto deja pasar demasiados
  falsos negativos (talentos reales no detectados) por el desbalance de clases — de ahi la recomendacion de
  operar con un umbral mas bajo y tratar la salida como una lista corta priorizada, no como veredicto final.
- En conjunto, Fase 2 (clustering) responde "¿que arquetipos de jugador existen y quien es reemplazable por
  quien a menor costo?" y Fase 3 (supervisado) responde "¿cuanto vale este jugador y que tan probable es que
  se consolide?" — ambos entregables alimentan directamente decisiones de fichaje, venta y renovacion del
  consorcio, cerrando el ciclo de la estrategia "Moneyball Avanzada" planteada en el objetivo del proyecto.